# Stack Overflow Salary Analysis
### Portfolio refresh of a 2019 Thinkful capstone

This notebook refreshes the original 2019 coursework using the **2018 Stack Overflow Developer Survey** while preserving the original analysis logic. The original notebook, `Stackoverflow_users_salary_prediction.ipynb`, remains unchanged.

The work is observational: reported associations do not establish causality.


## Data source and provenance

Stack Overflow's 2018 public release contains **98,855 qualified responses**. The original coursework filtered `CurrencySymbol == "USD"`; this means **salary reported in U.S. dollars**, not verified U.S. residence.

The coursework annualized monthly salary ×12 and weekly salary ×52. Stack Overflow's published 2018 methodology used 50 working weeks; this notebook keeps the original 52-week rule for provenance.

The original salary filter was **>$50,000 and <$195,000**.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
%matplotlib inline

DATA_CANDIDATES = [
    Path("data/survey_results_public.csv"),
    Path("../../../data/survey_results_public.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place the official 2018 survey_results_public.csv in the repository data/ folder."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
assert len(df) == 98_855, f"Expected 98,855 rows, found {len(df):,}"
print(f"Loaded {len(df):,} responses from {DATA_PATH}")


## Data preparation

The code below reproduces the original selected fields, USD filter, education mapping, complete-case deletion, salary annualization, and salary bounds.


In [ ]:
analysis_columns = [
    "Age", "Gender", "Employment", "Salary", "SalaryType",
    "CurrencySymbol", "YearsCodingProf", "FormalEducation",
    "WakeTime", "HoursComputer",
]
missing = sorted(set(analysis_columns) - set(df.columns))
assert not missing, f"Missing expected 2018 fields: {missing}"

education_labels = {
    "Some college/university study without earning a degree": "C W/O D",
    "Bachelor’s degree (BA, BS, B.Eng., etc.)": "BA",
    "Master’s degree (MA, MS, M.Eng., MBA, etc.)": "MA",
    "Associate degree": "AA",
    "Other doctoral degree (Ph.D, Ed.D., etc.)": "Ph.D",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "SecondarySchool",
    "Professional degree (JD, MD, etc.)": "ProDegree",
    "Primary/elementary school": "PrimarySchool",
    "I never completed any formal education": "NoEducation",
}

df_new = df[analysis_columns].copy()
df_new = df_new[df_new["CurrencySymbol"] == "USD"].copy()
df_new["FormalEducation"] = df_new["FormalEducation"].map(education_labels)
df_new["Salary"] = pd.to_numeric(
    df_new["Salary"].astype(str).str.replace(",", "", regex=False),
    errors="coerce",
)
df_new = df_new.dropna().copy()

# Preserved notebook checkpoint before salary bounds.
assert len(df_new) == 10_975, (
    f"Expected the original 10,975 complete USD-reporting rows, found {len(df_new):,}"
)

multiplier = {"Yearly": 1, "Monthly": 12, "Weekly": 52}
df_new["Converted_Salary"] = (
    df_new["Salary"] * df_new["SalaryType"].map(multiplier)
)
df_new = df_new[
    (df_new["Converted_Salary"] > 50_000)
    & (df_new["Converted_Salary"] < 195_000)
].copy()

print(f"Filtered analysis sample: {len(df_new):,}")
df_new["Converted_Salary"].describe()


## Age and professional experience

The original project asked whether salary varied with age and professional coding experience. For portfolio presentation, medians are used in the charts because salary distributions are typically skewed; this is a presentation refinement rather than an exact reproduction of the original mean plots.


In [ ]:
age_order = [
    "Under 18 years old", "18 - 24 years old", "25 - 34 years old",
    "35 - 44 years old", "45 - 54 years old", "55 - 64 years old",
    "65 years or older",
]
experience_order = [
    "0-2 years", "3-5 years", "6-8 years", "9-11 years", "12-14 years",
    "15-17 years", "18-20 years", "21-23 years", "24-26 years",
    "27-29 years", "30 or more years",
]

for field, order, title in [
    ("Age", age_order, "Median Annualized Salary by Age Group"),
    ("YearsCodingProf", experience_order, "Median Salary by Professional Coding Experience"),
]:
    med = (
        df_new.groupby(field)["Converted_Salary"].median()
        .reindex([x for x in order if x in df_new[field].unique()])
        .dropna()
    )
    ax = med.plot(kind="bar", figsize=(9, 4), title=title)
    ax.set_ylabel("Median salary (USD)")
    plt.tight_layout()
    plt.show()


Older and more experienced groups tended to show higher compensation in the original analysis. Age and experience are related, so these patterns should not be interpreted as independent causal effects.


## Coursework-defined composite segment

The original segment counted a respondent only when all three conditions were met:

- education was **BA, MA, or Ph.D**;
- wake time was before 5:00 AM through 8:00 AM;
- computer time was 9–12 hours or over 12 hours.

The original labels were subjective, so this copy uses neutral labels while preserving membership exactly.


In [ ]:
higher_education = {"BA", "MA", "Ph.D"}
early_wake = {
    "Before 5:00 AM", "Between 5:00 - 6:00 AM",
    "Between 6:01 - 7:00 AM", "Between 7:01 - 8:00 AM",
}
long_hours = {"9 - 12 hours", "Over 12 hours"}

df_new["CompositeSegment"] = np.where(
    df_new["FormalEducation"].isin(higher_education)
    & df_new["WakeTime"].isin(early_wake)
    & df_new["HoursComputer"].isin(long_hours),
    "DefinedSegment",
    "OtherRespondents",
)

defined = df_new.loc[
    df_new["CompositeSegment"] == "DefinedSegment", "Converted_Salary"
]
other = df_new.loc[
    df_new["CompositeSegment"] == "OtherRespondents", "Converted_Salary"
]
segment_test = ttest_ind(defined, other, equal_var=False, nan_policy="omit")

print(df_new.groupby("CompositeSegment")["Converted_Salary"].agg(["count", "mean", "median"]))
print(f"Welch p-value: {segment_test.pvalue:.12g}")
print("Preserved 2019 p-value: 0.8098447569123316")


The preserved 2019 output reported **p ≈ 0.8098**, providing no statistical evidence of a mean-salary difference between the constructed groups. Because education is embedded in the segment, it is best viewed as an exploratory feature-engineering exercise, not a validated “lifestyle” measure.


## Formal education and compensation

The original project compared BA, MA, and Ph.D groups with pairwise Welch t-tests.


In [ ]:
degree_data = {
    degree: df_new.loc[df_new["FormalEducation"] == degree, "Converted_Salary"]
    for degree in ["BA", "MA", "Ph.D"]
}
comparisons = [("BA", "MA"), ("Ph.D", "MA"), ("Ph.D", "BA")]

rows = []
for left, right in comparisons:
    result = ttest_ind(
        degree_data[left], degree_data[right],
        equal_var=False, nan_policy="omit",
    )
    rows.append({
        "Comparison": f"{left} vs {right}",
        "t_statistic": result.statistic,
        "p_value": result.pvalue,
    })
pd.DataFrame(rows)


### Preserved 2019 executed results

- **BA vs MA:** t = -9.466924020382079, p = 6.930610610508487e-21
- **Ph.D vs MA:** t = 4.136809621029714, p = 4.843046941003592e-05
- **Ph.D vs BA:** t = 8.182677672227504, p = 2.8459703132097765e-14

All three comparisons were statistically significant in the original output. They are unadjusted observational comparisons and do not control for experience, role, geography, specialty, or other confounders, so they do not establish that obtaining a degree causes higher salary.


## Computer hours among full-time respondents


In [ ]:
full_time = df_new[df_new["Employment"] == "Employed full-time"].copy()
hour_order = [
    "Less than 1 hour", "1 - 4 hours", "5 - 8 hours",
    "9 - 12 hours", "Over 12 hours",
]
median_by_hours = (
    full_time.groupby("HoursComputer")["Converted_Salary"].median()
    .reindex([x for x in hour_order if x in full_time["HoursComputer"].unique()])
    .dropna()
)
ax = median_by_hours.plot(
    kind="bar", figsize=(8, 4),
    title="Median Salary by Computer Hours — Full-Time Respondents",
)
ax.set_ylabel("Median annualized salary (USD)")
plt.tight_layout()
plt.show()


The descriptive pattern does not establish a simple “more computer hours = more salary” relationship. Computer hours may reflect role type, work style, or non-work computer use.


## Limitations and next steps

Key limitations are the self-selected sample, self-reported compensation, the fact that USD is not a geographic filter, manually chosen $50,000–$195,000 salary bounds, the coursework's 52-week annualization rather than Stack Overflow's 50-week method, possible confounding, unadjusted multiple pairwise tests, and no multivariable model.

A modern extension could use `Country` for geographic claims, compare both annualization methods, use systematic outlier sensitivity analyses, report confidence intervals and effect sizes, adjust for multiple testing, and build a multivariable model.

---

**Survey:** 2018 Stack Overflow Developer Survey  
**Original work:** Thinkful Data Science capstone, 2019  
**Original notebook preserved:** `Stackoverflow_users_salary_prediction.ipynb`
